<a href="https://colab.research.google.com/github/OlhaZahrebelna/Recommendation-Systems-Goodbooks-10k/blob/main/%22_RecSys_Goodbooks_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [1]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
# !pip -q install kaggle
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
# !kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

In [2]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings.csv не знайдено локально — завантажую з GitHub...
books.csv не знайдено локально — завантажую з GitHub...
book_tags.csv не знайдено локально — завантажую з GitHub...
tags.csv не знайдено локально — завантажую з GitHub...
ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [3]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [4]:
TOP_BOOKS = 1500      # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000        # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 140,934 | користувачів: 2,000 | книг: 1,496
Щільність: 0.0471


In [5]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 77,070 | користувачів з val-лайками: 1,991


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [6]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [7]:
import torch.nn.functional as F

item_emb = F.normalize(item_feats, dim=1)  # (M, n_genres) -> (M, n_genres)

In [8]:
def user_vector(user_idx):
    """
    user_idx -> L2-нормалізований вектор користувача.

    Вектор користувача = зважене середнє ембедингів книг,
    які користувач лайкнув у train.
    Вага = rating.
    """
    user_id = users[user_idx]

    user_hist = train_df[
        (train_df["user_id"] == user_id) &
        (train_df["rating"] >= LIKE_THRESHOLD)
    ]

    if len(user_hist) == 0:
        return F.normalize(item_emb.mean(dim=0), dim=0)

    item_indices = torch.tensor(
        [item_to_idx[b] for b in user_hist["book_id"]],
        dtype=torch.long
    )

    weights = torch.tensor(
        user_hist["rating"].values,
        dtype=torch.float32
    )

    emb = item_emb[item_indices]  # (n_user_likes, n_genres)

    user_vec = (emb * weights.unsqueeze(1)).sum(dim=0) / weights.sum()

    # Нормалізуємо, щоб dot product з item_emb був cosine similarity
    user_vec = F.normalize(user_vec.unsqueeze(0), p=2, dim=1).squeeze(0)

    return user_vec


In [9]:
def vsm_scores(user_idxs):
    """
    user_idxs -> матриця cosine-score для всіх книг.

    Повертає:
    scores shape = (len(user_idxs), M)
    """
    user_vecs = torch.stack([
        user_vector(int(u)) for u in user_idxs
    ])  # (n_users, n_genres)

    scores = user_vecs @ item_emb.T  # (n_users, M)

    return scores

**Питання:** Recall@10 у векторного підходу досить низький. Чому?


In [10]:
recall10 = recall_at_k(vsm_scores, k=10)
print(f"Recall@10 = {recall10:.4f}")

Recall@10 = 0.0522


In [11]:
# Візьмемо першого користувача
user_idx = 0

# Обчислюємо оцінки для всіх книг
scores = vsm_scores(torch.tensor([user_idx]))[0].clone()

# Прибираємо книги, які користувач уже бачив у train
for item_idx in seen_by_user[user_idx]:
    scores[item_idx] = -1e9

# Знаходимо топ-5 рекомендацій
top5_idx = torch.topk(scores, k=5).indices.tolist()

print(f"Топ-5 рекомендацій для користувача {users[user_idx]}:\n")
for rank, idx in enumerate(top5_idx, start=1):
    book_id = items[idx]
    title = title_of.get(book_id, "Невідома книга")
    print(f"{rank}. {title}")

Топ-5 рекомендацій для користувача 9:

1. Stardust
2. The Clan of the Cave Bear (Earth's Children, #1)
3. The Alchemist
4. Inkheart (Inkworld, #1)
5. Sophie's World


Recall@10 у векторного підходу низький, бо модель використовує лише 12 жанрових ознак, які дуже грубо описують книги. Вона не враховує автора, стиль, популярність, середній рейтинг, повний набір тегів та поведінку схожих користувачів.

Крім того, дані розріджені: кожен користувач оцінив лише малу частину всіх книг. Валідаційні лайки також не є повним списком усього, що користувач міг би вподобати, тому навіть хороша рекомендація може не зарахуватися в Recall@10.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class UserTower(nn.Module):
    def __init__(self, n_users, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, user_idx):
        user_vec = self.user_emb(user_idx)
        return F.normalize(self.mlp(user_vec), dim=1)


class ItemTower(nn.Module):
    def __init__(self, n_genres, emb_dim):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(n_genres, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, item_genres):
        item_vec = self.mlp(item_genres.float())
        return F.normalize(item_vec, dim=1)


class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=64, temperature=10.0):
        super().__init__()
        self.user_tower = UserTower(n_users, emb_dim)
        self.item_tower = ItemTower(n_genres, emb_dim)
        self.temperature = temperature

    def forward(self, user_idx, item_genres):
        user_vec = self.user_tower(user_idx)
        item_vec = self.item_tower(item_genres)

        logits = (user_vec * item_vec).sum(dim=1)
        return logits * self.temperature


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TwoTower(
    n_users=len(users),
    n_genres=n_genres,
    emb_dim=64,
    temperature=10.0
).to(device)

item_feats_device = item_feats.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

batch_size = 4096
epochs = 10
num_pos = len(pos_u)

In [14]:
for epoch in range(epochs):
    perm = torch.randperm(num_pos)

    total_loss = 0.0
    n_batches = 0

    for start in range(0, num_pos, batch_size):
        idx = perm[start:start + batch_size]

        u_pos = pos_u[idx].to(device)
        i_pos = pos_i[idx].to(device)

        # negative sampling: випадкова книга з усього корпусу
        i_neg = torch.randint(
            low=0,
            high=M,
            size=i_pos.shape,
            device=device
        )

        # users дублюємо: для positive і negative прикладів
        batch_users = torch.cat([u_pos, u_pos], dim=0)

        # item features для positive і negative книг
        batch_items = torch.cat([i_pos, i_neg], dim=0)
        batch_item_feats = item_feats_device[batch_items]

        # labels: 1 для positive, 0 для negative
        labels = torch.cat([
            torch.ones_like(u_pos, dtype=torch.float32),
            torch.zeros_like(u_pos, dtype=torch.float32)
        ], dim=0)

        logits = model(batch_users, batch_item_feats)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    print(f"Epoch {epoch + 1}/{epochs} | loss = {total_loss / n_batches:.4f}")

Epoch 1/10 | loss = 0.7247
Epoch 2/10 | loss = 0.6814
Epoch 3/10 | loss = 0.6731
Epoch 4/10 | loss = 0.6684
Epoch 5/10 | loss = 0.6634
Epoch 6/10 | loss = 0.6573
Epoch 7/10 | loss = 0.6492
Epoch 8/10 | loss = 0.6413
Epoch 9/10 | loss = 0.6311
Epoch 10/10 | loss = 0.6203


In [15]:
model.eval()

with torch.no_grad():
    item_vecs = model.item_tower(item_feats_device).cpu()

In [16]:
def two_tower_scores(user_idxs):
    model.eval()

    with torch.no_grad():
        user_idxs = user_idxs.to(device)
        user_vecs = model.user_tower(user_idxs).cpu()

        scores = user_vecs @ item_vecs.T

    return scores

In [17]:
tt_recall_10 = recall_at_k(two_tower_scores, k=10)
print(f"Two-Tower Recall@10: {tt_recall_10:.4f}")

Two-Tower Recall@10: 0.0476


In [18]:
user_idx = 0

torch.manual_seed(42)
scores = two_tower_scores(torch.tensor([user_idx]))[0].clone()

# прибираємо книги, які користувач уже бачив
for item_idx in seen_by_user[user_idx]:
    scores[item_idx] = -1e9

top5 = torch.topk(scores, k=5)

print(f"Топ-5 Two-Tower рекомендацій для користувача {users[user_idx]}:\n")

for rank, (idx, score) in enumerate(
    zip(top5.indices.tolist(), top5.values.tolist()),
    start=1
):
    book_id = items[idx]
    title = title_of.get(book_id, "Невідома книга")
    print(f"{rank}. {title} (score = {score:.4f})")

Топ-5 Two-Tower рекомендацій для користувача 9:

1. The Host (The Host, #1) (score = 0.1911)
2. The Hunger Games Trilogy Boxset (The Hunger Games, #1-3) (score = 0.1911)
3. Cloud Atlas (score = 0.1667)
4. A Storm of Swords (A Song of Ice and Fire, #3) (score = 0.1485)
5. Inkheart (Inkworld, #1) (score = 0.1347)


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [19]:
class NCF(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=64):
        super().__init__()

        self.user_emb = nn.Embedding(n_users, emb_dim)

        self.item_tower = nn.Sequential(
            nn.Linear(n_genres, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU()
        )

        self.mlp = nn.Sequential(
            nn.Linear(2 * emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, 1)
        )

    def forward(self, user_idx, item_genres):
        user_vec = self.user_emb(user_idx)
        item_vec = self.item_tower(item_genres.float())

        concat = torch.cat([user_vec, item_vec], dim=1)

        logits = self.mlp(concat).squeeze(1)
        return logits

Навчання


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ncf = NCF(
    n_users=len(users),
    n_genres=n_genres,
    emb_dim=64
).to(device)

item_feats_device = item_feats.to(device)

optimizer = torch.optim.Adam(ncf.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

batch_size = 4096
epochs = 10
num_pos = len(pos_u)

In [21]:
for epoch in range(epochs):
    perm = torch.randperm(num_pos)

    total_loss = 0.0
    n_batches = 0

    ncf.train()

    for start in range(0, num_pos, batch_size):
        idx = perm[start:start + batch_size]

        u_pos = pos_u[idx].to(device)
        i_pos = pos_i[idx].to(device)

        i_neg = torch.randint(
            low=0,
            high=M,
            size=i_pos.shape,
            device=device
        )

        batch_users = torch.cat([u_pos, u_pos], dim=0)
        batch_items = torch.cat([i_pos, i_neg], dim=0)
        batch_item_feats = item_feats_device[batch_items]

        labels = torch.cat([
            torch.ones_like(u_pos, dtype=torch.float32),
            torch.zeros_like(u_pos, dtype=torch.float32)
        ], dim=0)

        logits = ncf(batch_users, batch_item_feats)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    print(f"Epoch {epoch + 1}/{epochs} | loss = {total_loss / n_batches:.4f}")

Epoch 1/10 | loss = 0.6919
Epoch 2/10 | loss = 0.6834
Epoch 3/10 | loss = 0.6780
Epoch 4/10 | loss = 0.6747
Epoch 5/10 | loss = 0.6705
Epoch 6/10 | loss = 0.6662
Epoch 7/10 | loss = 0.6614
Epoch 8/10 | loss = 0.6559
Epoch 9/10 | loss = 0.6498
Epoch 10/10 | loss = 0.6422


In [22]:
def rank_ncf(user_idx, candidate_idxs):
    """
    user_idx: int
    candidate_idxs: list[int] або torch.Tensor з індексами книг

    Повертає список (item_idx, score), відсортований за спаданням score.
    """
    ncf.eval()

    if not torch.is_tensor(candidate_idxs):
        candidate_idxs = torch.tensor(candidate_idxs, dtype=torch.long)

    candidate_idxs = candidate_idxs.to(device)

    user_batch = torch.full(
        size=(len(candidate_idxs),),
        fill_value=user_idx,
        dtype=torch.long,
        device=device
    )

    candidate_feats = item_feats_device[candidate_idxs]

    with torch.no_grad():
        logits = ncf(user_batch, candidate_feats)
        scores = torch.sigmoid(logits)

    order = torch.argsort(scores, descending=True)

    ranked = [
        (candidate_idxs[i].item(), scores[i].item())
        for i in order
    ]

    return ranked

In [23]:
user_idx = 0

torch.manual_seed(42)
candidate_idxs = torch.randperm(M)[:5000].tolist()

# прибираємо вже бачені книги
candidate_idxs = [
    i for i in candidate_idxs
    if i not in seen_by_user[user_idx]
]

ranked = rank_ncf(user_idx, candidate_idxs)

print(f"Топ-5 NCF рекомендацій для користувача {users[user_idx]}:\n")

for rank, (item_idx, score) in enumerate(ranked[:5], start=1):
    book_id = items[item_idx]
    title = title_of.get(book_id, "Невідома книга")
    print(f"{rank}. {title} (score = {score:.4f})")

Топ-5 NCF рекомендацій для користувача 9:

1. Angels & Demons  (Robert Langdon, #1) (score = 0.8636)
2. Peace Like a River (score = 0.8479)
3. To Kill a Mockingbird (score = 0.8479)
4. Holes (Holes, #1) (score = 0.8448)
5. The Host (The Host, #1) (score = 0.8393)


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [24]:
def retrieve(user_idx, n_candidates):
    """
    Повертає список (item_idx, score), відсортований за спаданням Two-Tower score.
    """
    scores = two_tower_scores(torch.tensor([user_idx]))[0].clone()

    for item_idx in seen_by_user[user_idx]:
        scores[item_idx] = -1e9

    top = torch.topk(scores, k=n_candidates)

    return [
        (idx, score)
        for idx, score in zip(top.indices.tolist(), top.values.tolist())
    ]


def recommend_pipeline(user_idx, n_candidates, top_k):
    """
    Retrieval через Two-Tower → Ranking через NCF.
    """
    retrieved = retrieve(user_idx, n_candidates)

    candidate_idxs = [item_idx for item_idx, _ in retrieved]

    ranked = rank_ncf(user_idx, candidate_idxs)

    return ranked[:top_k]

In [25]:
for user_idx in [0, 1, 2]:
    print("=" * 80)
    print(f"Користувач: {users[user_idx]}")

    retrieved = retrieve(user_idx, n_candidates=10)

    print("\nRetrieval top-10:")
    for rank, (item_idx, score) in enumerate(retrieved, start=1):
        book_id = items[item_idx]
        title = title_of.get(book_id, "Невідома книга")
        print(f"{rank}. {title} | retrieval score = {score:.4f}")

    final_recs = recommend_pipeline(
        user_idx=user_idx,
        n_candidates=100,
        top_k=5
    )

    print("\nRanking top-5:")
    for rank, (item_idx, score) in enumerate(final_recs, start=1):
        book_id = items[item_idx]
        title = title_of.get(book_id, "Невідома книга")
        print(f"{rank}. {title} | NCF score = {score:.4f}")

Користувач: 9

Retrieval top-10:
1. The Hunger Games Trilogy Boxset (The Hunger Games, #1-3) | retrieval score = 0.1911
2. The Host (The Host, #1) | retrieval score = 0.1911
3. Cloud Atlas | retrieval score = 0.1667
4. A Storm of Swords (A Song of Ice and Fire, #3) | retrieval score = 0.1485
5. Inkheart (Inkworld, #1) | retrieval score = 0.1347
6. The Memory Keeper's Daughter | retrieval score = 0.1264
7. The Princess Bride  | retrieval score = 0.1256
8. Peace Like a River | retrieval score = 0.1208
9. To Kill a Mockingbird | retrieval score = 0.1208
10. Holes (Holes, #1) | retrieval score = 0.1151

Ranking top-5:
1. Angels & Demons  (Robert Langdon, #1) | NCF score = 0.8636
2. Peace Like a River | NCF score = 0.8479
3. To Kill a Mockingbird | NCF score = 0.8479
4. Holes (Holes, #1) | NCF score = 0.8448
5. The Host (The Host, #1) | NCF score = 0.8393
Користувач: 23

Retrieval top-10:
1. Catching Fire (The Hunger Games, #2) | retrieval score = 0.1816
2. The Hunger Games (The Hunger Game

**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Ми єкономимо час та результати будуть точнішими, якщо використовувати двоетапний спосіб.

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


### 1. Чому Recall@10 такий низький?

На реальних даних усі моделі цього домашнього завдання демонструють відносно низький Recall@10 з кількох причин. По-перше, використовуються досить бідні контентні ознаки книг — фактично лише жанри, причому їх небагато, тому модель не може точно розрізняти схожі книги. Наприклад, дві книги можуть належати до жанру фентезі, але одна бути дитячою, а інша — орієнтованою на дорослу аудиторію.

По-друге, дані є дуже розрідженими: кожен користувач взаємодіє лише з невеликою частиною всіх книг. Крім того, валідаційний набір містить не всі книги, які могли б сподобатися користувачу. Якщо модель не рекомендувала певну книгу, користувач її не прочитав і не оцінив, хоча вона могла б бути для нього релевантною.

### 2. Як покращити якість, не змінюючи архітектуру?

Якість можна підвищити, додавши більше інформативних ознак книг і користувачів. Наприклад:

* автора книги;
* рік публікації;
* середній рейтинг;
* розширений набір тегів (не лише існуючі жанри, а й додаткові теги, представлені через TF-IDF);
* текстові ембединги опису книги (наприклад, отримані за допомогою BERT).

Ці ознаки дозволять моделі краще розрізняти схожі книги та точніше враховувати вподобання користувачів.

### 3. Diversity

Якщо користувач полюбляє фентезі, не варто рекомендувати йому десять фентезі-книг поспіль, оскільки стрічка виглядатиме одноманітною. Для підвищення різноманітності можна застосовувати reranking після ранжування рекомендацій або обмежувати максимальну кількість книг одного жанру в топі, підмішуючи книги інших тематик.

### 4. Freshness / Cold Start

Нова книга, яка ще не має жодної оцінки, може бути рекомендована контентно-орієнтованими підходами, наприклад моделлю two-tower, якщо для неї доступні описові ознаки (жанри, теги, автор тощо). Натомість методи, що ґрунтуються лише на колаборативній фільтрації, не зможуть рекомендувати таку книгу одразу, оскільки їм потрібні взаємодії користувачів із нею.

### 5. Watch Time > CTR

YouTube оптимізує час перегляду (Watch Time), а не лише CTR, тому що клік не гарантує корисності рекомендації. Користувач може натиснути на відео, але закрити його вже через кілька секунд. Натомість довший час перегляду свідчить про те, що контент дійсно зацікавив користувача.

Технічно це враховується у weighted logistic regression: приклади з більшим часом перегляду отримують більшу вагу під час навчання, тому модель навчається віддавати перевагу рекомендаціям, які не лише приваблюють кліком, а й утримують увагу користувача.
